**© Copyright AIDENTIFY. All rights reserved.**

# Part 1 | Session 09: LoRA vs FFT 실전 비교 실습

## 🎯 LoRA vs FFT 실전 비교 실습

이전 세션에서 이론적 비교를 했다면, 이번 세션에서는 **동일한 데이터**로 LoRA와 FFT를 실제 학습하고 **추론 품질까지 비교**합니다.

### 실험 설계

| 항목 | LoRA | FFT |
|------|------|-----|
| 모델 | Qwen2.5-1.5B (bf16) | Qwen2.5-1.5B (fp32 + bf16 mixed precision) |
| 데이터 | Alpaca 한국어 샘플 (동일) | Alpaca 한국어 샘플 (동일) |
| 에포크 | 3 | 3 |
| batch_size | 4 (accum 2) | 2 (accum 4) |
| 유효 배치 | 8 | 8 |
| 평가 | 동일 프롬프트 5개 | 동일 프롬프트 5개 |

### 학습 목표

- ✅ 동일 조건에서 LoRA와 FFT의 실제 성능 차이 확인
- ✅ 학습 곡선(loss) 비교
- ✅ 추론 품질 비교
- ✅ 종합적인 비용-효과 분석

## 1️⃣ 공통 설정 (데이터, 평가 기준)

In [1]:
# 필수 라이브러리
import torch
import gc
import os
import json
import time

# 참고: Part 1 실습은 단일 GPU 기준이라 DeepSpeed를 사용하지 않는다.
# deepspeed 패키지가 설치돼 있으면 accelerate가 PEFT 모델 + gradient_checkpointing
# 경로에서 이를 import하는데, CUDA 툴킷(nvcc/CUDA_HOME)이 없는 환경에서는
# MissingCUDAException으로 학습이 시작조차 되지 않는다. venv에서 제거해 두었다.

# CUDA 메모리 단편화 완화 (8GB GPU에서 FFT 학습 시 OOM 여유 확보).
# CUDA 초기화 이전(첫 텐서 할당 전)에 설정해야 적용되므로 여기서 지정한다.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

print("✅ 라이브러리 임포트 완료")
print(f"CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

✅ 라이브러리 임포트 완료
CUDA: True, GPU: NVIDIA GeForce RTX 3090


In [2]:
# GPU 메모리 모니터링 함수
def print_gpu_memory(tag=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU: {allocated:.1f}GB / {total:.1f}GB")

def get_gpu_memory_gb():
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024**3
    return 0

# 공통 설정
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LENGTH = 1024

# 평가용 테스트 프롬프트 (특수한 상황 — 특정 인물/설정을 부여하고 그 맥락에서 응답하게 함)
TEST_PROMPTS = [
    # 1) 인물 설정 + 소개글 생성
    "김의중은 인공지능을 강의하는 강사입니다. 그를 소개하는 짧은 강사 프로필을 작성해주세요.",
    # 2) 역할 부여 + 상황 대응
    "당신은 RTX 4060 한 대만 가진 대학원생의 멘토입니다. 제한된 GPU로 LLM을 파인튜닝하려는 학생에게 조언해주세요.",
    # 3) 가상의 제품 설정 + 설명
    "'한글로'라는 한국어 특화 오픈소스 LLM이 새로 출시되었다고 가정하고, 이 모델의 보도자료를 작성해주세요.",
    # 4) 특정 청중 설정 + 눈높이 설명
    "초등학생에게 LoRA 파인튜닝이 무엇인지 비유를 들어 쉽게 설명해주세요.",
    # 5) 가상 시나리오 + 문제 해결
    "스타트업 개발자인 당신은 고객 상담 챗봇을 만들어야 합니다. 8GB GPU 환경에서 LoRA로 챗봇을 학습시키는 단계별 계획을 세워주세요."
]

print(f"📌 모델: {MODEL_NAME}")
print(f"📌 테스트 프롬프트 수: {len(TEST_PROMPTS)}")
print_gpu_memory("시작")

📌 모델: Qwen/Qwen2.5-1.5B-Instruct
📌 테스트 프롬프트 수: 5
[시작] GPU: 0.0GB / 23.6GB


In [3]:
# 공통 데이터 로드 및 전처리
# 특수한 상황 평가를 위해 페르소나/설정 지식(김의중 강사, 가상 모델 '한글로' 등)을
# 주입하는 데이터셋을 사용한다. 베이스 모델이 모르는 사실이므로 학습 전/후 차이가 뚜렷하게 드러난다.
data_path = "data/samples/persona_ko_sample.json"

with open(data_path, "r", encoding="utf-8") as f:
    alpaca_data = json.load(f)

# 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Alpaca → Chat Template 변환
def format_alpaca_to_text(sample):
    instruction = sample["instruction"]
    input_text = sample.get("input", "")
    output_text = sample["output"]
    
    user_content = f"{instruction}\n\n{input_text}" if input_text else instruction
    messages = [
        {"role": "system", "content": "당신은 유용한 AI 어시스턴트입니다."},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": output_text}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

formatted_texts = [format_alpaca_to_text(s) for s in alpaca_data]
dataset = Dataset.from_dict({"text": formatted_texts})

print(f"✅ 데이터 준비 완료: {len(dataset)}개 샘플")

# 학습 결과 저장용
results = {"lora": {}, "fft": {}}

✅ 데이터 준비 완료: 20개 샘플


In [4]:
# 추론 함수
def generate_responses(model, tokenizer, prompts, max_new_tokens=150):
    """모델로 응답을 생성하는 공통 함수"""
    model.eval()

    
    responses = []
    for prompt in prompts:
        messages = [
            {"role": "system", "content": "당신은 유용한 AI 어시스턴트입니다."},
            {"role": "user", "content": prompt}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=False, temperature=1.0
            )
        response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        responses.append(response)

    return responses

print("✅ 공통 함수 정의 완료")

✅ 공통 함수 정의 완료


## 2️⃣ LoRA 학습 실행 (r=16)

In [5]:
print("="*60)
print("🔵 LoRA 학습 시작")
print("="*60)

# LoRA 모델 로드 (bf16, 양자화 없음)
# GPU 메모리가 충분한 환경(24GB급)이므로 4bit 양자화(QLoRA)를 쓰지 않는다.
# 양자화를 빼면 학습/추론이 빨라지고, FFT와 같은 정밀도에서 비교할 수 있다.
# 8GB급 GPU에서는 아래 주석의 4bit 설정으로 되돌리고
# gradient_checkpointing을 다시 켜면 된다.
#
#   bnb_config = BitsAndBytesConfig(
#       load_in_4bit=True, bnb_4bit_quant_type="nf4",
#       bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
#   lora_model = AutoModelForCausalLM.from_pretrained(
#       MODEL_NAME, quantization_config=bnb_config, device_map={"": 0}, trust_remote_code=True)
#   lora_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
#   lora_model.enable_input_require_grads()
lora_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16,
    device_map={"": 0}, trust_remote_code=True
)

# 학습 전 응답 저장
print("\n📋 학습 전 응답 수집 중...")
before_responses = generate_responses(lora_model, tokenizer, TEST_PROMPTS)

# LoRA 적용
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM
)
lora_model = get_peft_model(lora_model, lora_config)
lora_model.print_trainable_parameters()
print_gpu_memory("LoRA 모델 준비 완료")

🔵 LoRA 학습 시작


The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



📋 학습 전 응답 수집 중...
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
[LoRA 모델 준비 완료] GPU: 3.0GB / 23.6GB


In [6]:
# LoRA SFTTrainer 학습
lora_sft_config = SFTConfig(
    output_dir="./output/comparison_lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,   # 메모리 여유 -> 배치를 키워 스텝당 처리량 확보
    gradient_accumulation_steps=2,   # 유효 배치는 4x2=8로 기존과 동일
    learning_rate=2e-4,
    bf16=True,
    logging_steps=1,
    save_strategy="epoch",
   
    lr_scheduler_type="cosine",
    max_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    report_to="none",
    gradient_checkpointing=False,    # 메모리 여유 -> 재계산 대신 속도를 택한다
)

lora_trainer = SFTTrainer(
    model=lora_model, args=lora_sft_config,
    train_dataset=dataset, processing_class=tokenizer,
)

print("🚀 LoRA 학습 시작!")
lora_start = time.time()
lora_result = lora_trainer.train()
lora_time = time.time() - lora_start

# 결과 저장
results["lora"]["time"] = lora_time
results["lora"]["loss"] = lora_result.training_loss
results["lora"]["memory"] = (
    torch.cuda.max_memory_allocated() / 1024**3
)
results["lora"]["log_history"] = lora_trainer.state.log_history

print(f"\n✅ LoRA 학습 완료!")
print(f"📌 소요 시간: {lora_time:.1f}초")
print(f"📌 Final Loss: {lora_result.training_loss:.4f}")
print_gpu_memory("LoRA 학습 완료")

Adding EOS to train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


🚀 LoRA 학습 시작!


Step,Training Loss
1,2.839100
2,2.534600
3,2.437800
4,2.035700
5,1.939900
6,1.646700
7,1.687100
8,1.627900
9,1.604600



✅ LoRA 학습 완료!
📌 소요 시간: 8.9초
📌 Final Loss: 2.0393
[LoRA 학습 완료] GPU: 3.1GB / 23.6GB


In [7]:
# LoRA 학습 후 응답
print("\n📋 LoRA 학습 후 응답 수집 중...")
lora_after_responses = generate_responses(lora_model, tokenizer, TEST_PROMPTS)
results["lora"]["responses"] = lora_after_responses

# LoRA 어댑터 크기
lora_save_path = "./output/comparison_lora/lora_adapter"
lora_model.save_pretrained(lora_save_path)
lora_size = sum(
    os.path.getsize(os.path.join(root, f))
    for root, dirs, files in os.walk(lora_save_path)
    for f in files
)
results["lora"]["save_size"] = lora_size
print(f"📌 LoRA 어댑터 크기: {lora_size/1024/1024:.1f} MB")

# 메모리 해제
del lora_model, lora_trainer
gc.collect()
torch.cuda.empty_cache()
print("✅ LoRA 메모리 해제 완료")


📋 LoRA 학습 후 응답 수집 중...


📌 LoRA 어댑터 크기: 70.5 MB
✅ LoRA 메모리 해제 완료


## 3️⃣ FFT 학습 실행 (1.5B 모델)

⚠️ 1.5B 전체 파인튜닝은 24GB급 GPU가 필요합니다(fp32 가중치 + 그래디언트 + 8bit 옵티마이저 상태 ≈ 16GB, 활성값 포함 20GB 내외). RTX 4060(8GB)에서는 OOM이 나며, 코드가 에러를 잡아 LoRA와의 차이를 보여줍니다.

In [8]:
print("="*60)
print("🟠 FFT 학습 시작")
print("="*60)

try:
    # 전체 파인튜닝 모델 로드 (양자화 없음)
    # ⚠️ 여기서 float16/bfloat16 가중치를 그대로 올리면 안 된다.
    #    bf16=True는 GradScaler를 쓰지 않으므로 AdamW가 저정밀도 파라미터를 직접 갱신하는데,
    #    fp16에서는 2차 모멘트와 eps(1e-8)가 언더플로되어 grad/(sqrt(v)+eps)가 폭주한다.
    #    (실제 증상: loss 2.8 -> 21 -> 852 로 발산)
    #    → 마스터 가중치는 fp32로 두고, 연산만 bf16 autocast(bf16=True)로 처리한다.
    fft_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, dtype=torch.float32,
        device_map={"": 0}, trust_remote_code=True
    )
    # 학습 전 응답 (비교 기준으로 동일 모델)
    print_gpu_memory("FFT 모델 로드 후")
    
    # SFTTrainer 학습
    fft_sft_config = SFTConfig(
        output_dir="./output/comparison_fft",
        num_train_epochs=3,
        # FFT는 fp32 가중치(6GB)+그래디언트(6GB)+8bit 옵티마이저 상태(3GB)로 약 16GB가
        # 고정으로 잡혀 활성값 여유가 크지 않다(A10G 24GB 기준 실측 피크 19.1GB).
        # batch를 2로 올리고 체크포인팅을 끄면 20.9GB까지 올라가 OOM이 났으므로 batch 1을 유지한다.
        # GPU를 혼자 쓴다면 batch 2 + accum 4까지는 시도해볼 만하다.
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,   # 유효 배치 1x8=8 (LoRA와 동일 조건)
        learning_rate=2e-5,  # FFT는 더 작은 lr 사용
        bf16=True,           # fp32 마스터 가중치 + bf16 연산 (mixed precision)
        # ⚠️ FFT에서 save_strategy="epoch"를 쓰면 에폭마다 체크포인트가 13GB씩 쌓인다
        #    (fp32 가중치 6GB + 옵티마이저/스케줄러 상태). 3에폭이면 39GB.
        #    학습 후 모델은 셀 14에서 직접 저장하므로 중간 체크포인트는 남기지 않는다.
        save_strategy="no",
        # 1.5B fp32 전체 파인튜닝은 가중치 6GB + 그래디언트 6GB + AdamW 상태 12GB = 약 24GB라
        # 24GB GPU에서도 순정 AdamW는 들어가지 않는다. 옵티마이저 상태만 8bit로 줄여 ~3GB로 만든다.
        optim="adamw_bnb_8bit",
        max_grad_norm=1.0,
        logging_steps=1,
      
        lr_scheduler_type="cosine",
        max_length=MAX_SEQ_LENGTH,
        dataset_text_field="text",
        report_to="none",
        # FFT는 fp32 가중치+그래디언트+옵티마이저 상태만 ~16GB라 활성값 여유가 크지 않다.
        # 체크포인팅을 끄면 24GB에서도 OOM이 나므로 여기서는 켜 둔다.
        gradient_checkpointing=True,
    )
    
    fft_trainer = SFTTrainer(
        model=fft_model, args=fft_sft_config,
        train_dataset=dataset, processing_class=tokenizer,
    )
    
    print("🚀 FFT 학습 시작!")
    fft_start = time.time()
    fft_result = fft_trainer.train()
    fft_time = time.time() - fft_start
    
    results["fft"]["time"] = fft_time
    results["fft"]["loss"] = fft_result.training_loss
    results["fft"]["memory"] = get_gpu_memory_gb()
    results["fft"]["log_history"] = fft_trainer.state.log_history
    
    print(f"\n✅ FFT 학습 완료!")
    print(f"📌 소요 시간: {fft_time:.1f}초")
    print(f"📌 Final Loss: {fft_result.training_loss:.4f}")
    print_gpu_memory("FFT 학습 완료")
    
except RuntimeError as e:
    if "out of memory" in str(e).lower():
        print("\n⚠️ GPU 메모리 부족으로 FFT 학습 실패!")
        print("📌 이것이 바로 LoRA가 필요한 이유입니다.")
        print("📌 RTX 4060(8GB)에서 1.5B FFT는 메모리가 매우 빠듯합니다.")
        results["fft"]["time"] = float('inf')
        results["fft"]["loss"] = float('inf')
        results["fft"]["memory"] = (
          torch.cuda.max_memory_allocated() / 1024**3
        )
        gc.collect()
        torch.cuda.empty_cache()
    else:
        raise e

🟠 FFT 학습 시작


[FFT 모델 로드 후] GPU: 5.8GB / 23.6GB


Adding EOS to train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


🚀 FFT 학습 시작!


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
1,2.829100
2,2.242200
3,2.028500
4,1.464900
5,1.138400
6,0.955100
7,0.759800
8,0.764000
9,0.767300



✅ FFT 학습 완료!
📌 소요 시간: 10.1초
📌 Final Loss: 1.4388
[FFT 학습 완료] GPU: 10.0GB / 23.6GB


In [9]:
# FFT 학습 후 응답 (학습 성공 시)
if results["fft"]["loss"] != float('inf'):
    print("📋 FFT 학습 후 응답 수집 중...")
    fft_after_responses = generate_responses(fft_model, tokenizer, TEST_PROMPTS)
    results["fft"]["responses"] = fft_after_responses
    
    # FFT 모델 크기
    fft_save_path = "./output/comparison_fft/full_model"
    os.makedirs(fft_save_path, exist_ok=True)

    # 모델 및 토크나이저 저장
    fft_model.save_pretrained(fft_save_path)

    # FFT 모델 폴더 크기 측정
    fft_size = sum(
        os.path.getsize(os.path.join(root, f))
        for root, dirs, files in os.walk(fft_save_path)
        for f in files
    )
    results["fft"]["save_size"] = fft_size
    print(f"📌 FFT 모델 크기: {fft_size/1024/1024:.0f} MB")
    
    del fft_model, fft_trainer
    gc.collect()
    torch.cuda.empty_cache()
else:
    results["fft"]["responses"] = ["(OOM - 학습 실패)"] * len(TEST_PROMPTS)
    results["fft"]["save_size"] = 3_000_000_000  # 추정값 ~3GB
    print("⚠️ FFT 학습이 실패하여 응답을 생성할 수 없습니다.")

print("✅ FFT 메모리 해제 완료")

📋 FFT 학습 후 응답 수집 중...


📌 FFT 모델 크기: 5889 MB
✅ FFT 메모리 해제 완료


## 4️⃣ 추론 품질 비교 (동일 프롬프트)

In [10]:
print("="*70)
print("📊 추론 품질 비교: Before vs LoRA vs FFT")
print("="*70)

for i, prompt in enumerate(TEST_PROMPTS):
    print(f"\n{'='*70}")
    print(f"🔹 질문 {i+1}: {prompt[:70]}")
    print(f"\n  [Before]  {before_responses[i][:200]}")
    print(f"\n  [LoRA]    {results['lora']['responses'][i][:200]}")
    print(f"\n  [FFT]     {results['fft']['responses'][i][:200]}")

print(f"\n{'='*70}")

📊 추론 품질 비교: Before vs LoRA vs FFT

🔹 질문 1: 김의중은 인공지능을 강의하는 강사입니다. 그를 소개하는 짧은 강사 프로필을 작성해주세요.

  [Before]  김의중 강사는 인공지능 분야에서 전문가이며, 다양한 기술과 트렌드에 대해 깊이 있는 이해를 가지고 있습니다. 그는 학습 알고리즘, 딥러닝, 자연어 처리 등 주요 주제에 대한 강력한 지식을 가집니다.

김의중 강사는 또한 실무 경험도 풍부합니다. 그는 여러 기업에서 인공지능 관련 업무를 수행하며, 실제 문제 해결 능력을 갖추고 있습니다. 이는 김의중 강사에게

  [LoRA]    김의중은 인공지능과 관련된 강의를 주로 진행하고 있습니다. 그는 대학원에서 인공지능에 대한 연구를 했으며, 현재는 인공지능 분야의 전문가로 활동하고 있습니다. 그는 다양한 학습 방법론과 알고리즘에 대해 설명하며, 특히 딥러닝과 머신러닝에 초점을 맞추고 있습니다. 또한, 김의중은 실무 경험도 가지고 있어, 실제 데이터셋을 활용한 구현 방법과 문제 해결 능력이

  [FFT]     김의중은 인공지능과 머신러닝에 특화된 강사입니다. 2017년부터 대규모 언어 모델(LLM)과 GPT-3, QLoRA와 LoRA를 주제로 강의하며, 2023년에는 LoRA와 QLoRA를 학습률과 파인튜닝으로 적용한 실습 중심 교육을 진행합니다.

🔹 질문 2: 당신은 RTX 4060 한 대만 가진 대학원생의 멘토입니다. 제한된 GPU로 LLM을 파인튜닝하려는 학생에게 조언해주세요.

  [Before]  RTX 4060이 주어진 상황에서 파인 튜닝(Large Language Model, LLM)에 대한 작업을 수행하는 것은 매우 어렵습니다. 이 GPU는 일반적으로 대형 모델을 실행하는데 적합하지 않습니다.

대신, 다음 방법들을 고려해보세요:

1. **GPU를 더 많은 컴퓨터나 클라우드 서비스로 분산**: 여러 컴퓨터 또는 클라우드 서비스를 사용하여 파인

  [LoRA]    RTX 4060은 12GB 메모리, 3550M

## 5️⃣ 학습 곡선(loss) 비교

In [11]:
# 학습 로그에서 loss 추출
def extract_losses(log_history):
    """학습 로그에서 loss 값만 추출"""
    losses = []
    steps = []
    for entry in log_history:
        if "loss" in entry:
            losses.append(entry["loss"])
            steps.append(entry.get("step", len(losses)))
    return steps, losses

lora_steps, lora_losses = extract_losses(results["lora"].get("log_history", []))

print("📊 LoRA 학습 곡선 (텍스트)")
print("="*50)
if lora_losses:
    for step, loss in zip(lora_steps, lora_losses):
        bar = "█" * int(loss * 10)
        print(f"  Step {step:3d}: {loss:.4f} {bar}")

if results["fft"]["loss"] != float('inf'):
    fft_steps, fft_losses = extract_losses(results["fft"].get("log_history", []))
    print(f"\n📊 FFT 학습 곡선 (텍스트)")
    print("="*50)
    if fft_losses:
        for step, loss in zip(fft_steps, fft_losses):
            bar = "█" * int(loss * 10)
            print(f"  Step {step:3d}: {loss:.4f} {bar}")
else:
    print("\n⚠️ FFT 학습이 실패하여 loss 곡선을 표시할 수 없습니다.")

📊 LoRA 학습 곡선 (텍스트)
  Step   1: 2.8391 ████████████████████████████
  Step   2: 2.5346 █████████████████████████
  Step   3: 2.4378 ████████████████████████
  Step   4: 2.0357 ████████████████████
  Step   5: 1.9399 ███████████████████
  Step   6: 1.6467 ████████████████
  Step   7: 1.6871 ████████████████
  Step   8: 1.6279 ████████████████
  Step   9: 1.6046 ████████████████

📊 FFT 학습 곡선 (텍스트)
  Step   1: 2.8291 ████████████████████████████
  Step   2: 2.2422 ██████████████████████
  Step   3: 2.0285 ████████████████████
  Step   4: 1.4649 ██████████████
  Step   5: 1.1384 ███████████
  Step   6: 0.9551 █████████
  Step   7: 0.7598 ███████
  Step   8: 0.7640 ███████
  Step   9: 0.7673 ███████


In [12]:
# matplotlib이 있으면 시각화
try:
    import matplotlib.pyplot as plt
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    
    if lora_losses:
        ax.plot(lora_steps, lora_losses, 'b-o', label='LoRA', markersize=3)
    
    if results["fft"]["loss"] != float('inf'):
        fft_steps, fft_losses = extract_losses(results["fft"].get("log_history", []))
        if fft_losses:
            ax.plot(fft_steps, fft_losses, 'r-o', label='FFT', markersize=3)
    
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.set_title('LoRA vs FFT Loss Curve Comparison')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("./output/loss_comparison.png", dpi=100)
    plt.show()
    print("✅ 그래프 저장: ./output/loss_comparison.png")
    
except ImportError:
    print("ℹ️ matplotlib이 없어 텍스트로만 표시합니다. (pip install matplotlib)")

ℹ️ matplotlib이 없어 텍스트로만 표시합니다. (pip install matplotlib)


## 6️⃣ 종합 비교표 (메모리, 시간, 성능, 저장 크기)

In [13]:
print("="*75)
print("📊 LoRA vs FFT 종합 비교 결과")
print("="*75)

# 파이썬 3.11 이하에서는 f-string 안에 같은 종류의 따옴표를 중첩할 수 없다.
# (PEP 701은 3.12부터) 그래서 표시할 문자열을 미리 만들어 둔다.
lora, fft = results["lora"], results["fft"]
fft_ok = fft["loss"] != float('inf')

fft_status = "성공" if fft_ok else "실패 (OOM)"
fft_time_str = f"{fft['time']:.1f}초" if fft_ok else "N/A (OOM)"
fft_loss_str = f"{fft['loss']:.4f}" if fft_ok else "N/A"
fft_mem_str = f"~{fft['memory']:.1f}GB"
fft_size_str = f"{fft['save_size'] / 1024 / 1024:.0f}MB"

lora_time_str = f"{lora['time']:.1f}초"
lora_loss_str = f"{lora['loss']:.4f}"
lora_mem_str = f"~{lora['memory']:.1f}GB"
lora_size_str = f"{lora['save_size'] / 1024 / 1024:.1f}MB"

print(f"\n{'항목':<25} {'LoRA':<25} {'FFT':<25}")
print("-"*75)
print(f"{'학습 상태':<25} {'✅ 성공':<25} {fft_status:<25}")
print(f"{'학습 시간':<25} {lora_time_str:<25} {fft_time_str:<25}")
print(f"{'Final Loss':<25} {lora_loss_str:<25} {fft_loss_str:<25}")
print(f"{'GPU 메모리':<25} {lora_mem_str:<25} {fft_mem_str:<25}")
print(f"{'저장 크기':<25} {lora_size_str:<25} {fft_size_str:<25}")
print(f"{'학습 파라미터':<25} {'~1%':<25} {'100%':<25}")
print(f"{'RTX 4060 적합도':<25} {'✅ 매우 적합':<25} {'⚠️ 빠듯/불가':<25}")
print("-"*75)

print(f"\n📌 결론:")
print(f"   RTX 4060(8GB) 환경에서는 LoRA가 안정적이고 실용적입니다.")
print(f"   FFT는 충분한 VRAM(24GB+)이 있을 때 고려하세요.")

📊 LoRA vs FFT 종합 비교 결과

항목                        LoRA                      FFT                      
---------------------------------------------------------------------------
학습 상태                     ✅ 성공                      성공                       
학습 시간                     8.9초                      10.1초                    
Final Loss                2.0393                    1.4388                   
GPU 메모리                   ~9.8GB                    ~10.0GB                  
저장 크기                     70.5MB                    5889MB                   
학습 파라미터                   ~1%                       100%                     
RTX 4060 적합도              ✅ 매우 적합                   ⚠️ 빠듯/불가                 
---------------------------------------------------------------------------

📌 결론:
   RTX 4060(8GB) 환경에서는 LoRA가 안정적이고 실용적입니다.
   FFT는 충분한 VRAM(24GB+)이 있을 때 고려하세요.


In [14]:
# 임시 파일 정리
import shutil
for path in ["./output/comparison_lora", "./output/comparison_fft"]:
    if os.path.exists(path):
        shutil.rmtree(path)
if os.path.exists("./output/loss_comparison.png"):
    os.remove("./output/loss_comparison.png")
print("✅ 임시 파일 정리 완료")

✅ 임시 파일 정리 완료


## 📝 정리 및 핵심 요약

### 이번 실습에서 배운 내용

- 🎯 **동일 데이터, 동일 조건**에서 LoRA와 FFT를 직접 비교했습니다
- 🎯 **LoRA는 메모리 효율적**이면서도 합리적인 성능을 달성합니다
- 🎯 **FFT는 더 많은 자원**이 필요하지만, 최고 성능을 낼 수 있습니다
- 🎯 **RTX 4060에서 FFT는 1.5B까지만** 시도할 수 있고, 빠듯합니다

### 실무 가이드

| GPU VRAM | 추천 방법 | 최대 모델 크기 |
|----------|----------|---------------|
| 8GB (RTX 4060) | QLoRA | 7B |
| 16GB (RTX 4080) | LoRA | 7B, QLoRA 13B |
| 24GB (RTX 4090) | LoRA/FFT | FFT 3B, LoRA 13B |
| 48GB+ (A6000, A100) | FFT | FFT 13B+ |

### 다음 단계

- ➡️ **Session 12c**: Unsloth 기반 파인튜닝 (LoRA 최적화)
- ➡️ **Session 13** (그 다음): 데이터 파이프라인 & 합성 데이터/Distillation